# <center> Deep Learning Coursework </center>
## <center> Audio Text Transformer Implementation </center>

Explaination of Audio Text Transformer

Imports required

In [3]:
from pathlib import Path
import torch
import torch.backends.cudnn
import numpy as np
from torch import nn, optim
from torch.nn import functional as F
import torchvision.datasets
from torch.optim.optimizer import Optimizer
from torch.utils.data import DataLoader
import torchvision.transforms as T
from torchvision import transforms

from audio_text_transformer import (
    DEVICE,
    AudioTextDataset,
    AudioTextTransformer,
    Trainer,
    get_summary_writer_log_dir,
    load_features,
    set_seed,
)

DATASET_ROOT = Path("/data/COMS30017")
LOG_DIR = Path("logs")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cpu


load data centres

In [4]:
set_seed(42)

train_audio, train_text, train_labels = load_features(DATASET_ROOT, "train")
val_audio, val_text, val_labels = load_features(DATASET_ROOT, "val")

train_dataset = AudioTextDataset(train_audio, train_text, train_labels, training=True)
val_dataset = AudioTextDataset(val_audio, val_text, val_labels, training=False)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

print("Train samples:", len(train_dataset), "| Val samples:", len(val_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '\\data\\COMS30017\\train_audio_features.pt'

baseline mean pooling

In [5]:
set_seed(42)
sample_a, sample_t, _ = next(iter(train_loader))

mean_model = AudioTextTransformer(
    audio_dim=sample_a.shape[-1],
    text_dim=sample_t.shape[-1],
    pooling="mean",
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mean_model.parameters(), lr=1e-3)

log_dir = get_summary_writer_log_dir(
    type("Args", (), {"log_dir": Path("logs"), "batch_size": 128, "learning_rate": 1e-3})()
)
writer = SummaryWriter(str(log_dir), flush_secs=5)

mean_trainer = Trainer(mean_model, train_loader, val_loader, criterion, optimizer, writer, DEVICE)
mean_trainer.train(epochs=10, val_frequency=2, print_frequency=10, log_frequency=10)
torch.save(mean_trainer.model.state_dict(), "weights_mean.pth")
writer.close()

NameError: name 'train_loader' is not defined

extension

In [ ]:
set_seed(42)
sample_a, sample_t, _ = next(iter(train_loader))

attn_model = AudioTextTransformer(
    audio_dim=sample_a.shape[-1],
    text_dim=sample_t.shape[-1],
    pooling="attention",
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(attn_model.parameters(), lr=1e-3)

log_dir = get_summary_writer_log_dir(
    type("Args", (), {"log_dir": Path("logs"), "batch_size": 128, "learning_rate": 1e-3})()
)
writer = SummaryWriter(str(log_dir), flush_secs=5)

attn_trainer = Trainer(attn_model, train_loader, val_loader, criterion, optimizer, writer, DEVICE)
attn_trainer.train(epochs=10, val_frequency=2, print_frequency=10, log_frequency=10)
torch.save(attn_trainer.model.state_dict(), "weights_attention.pth")
writer.close()

compare accuracy

In [ ]:
def evaluate_checkpoint(weights_path: Path, pooling: str, loader: DataLoader):
    sample_a, sample_t, _ = next(iter(loader))
    eval_model = AudioTextTransformer(
        audio_dim=sample_a.shape[-1],
        text_dim=sample_t.shape[-1],
        pooling=pooling,
    ).to(DEVICE)

    state_dict = torch.load(weights_path, map_location=DEVICE, weights_only=True)
    eval_model.load_state_dict(state_dict)
    eval_model.eval()

    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    preds_all, labels_all = [], []

    with torch.no_grad():
        for audio, text, labels in loader:
            audio = audio.to(DEVICE)
            text = text.to(DEVICE)
            labels = labels.to(DEVICE)

            logits = eval_model(audio, text)
            loss = criterion(logits, labels)
            total_loss += loss.item()

            preds_all.extend(logits.argmax(dim=-1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    acc = float(np.mean(np.array(preds_all) == np.array(labels_all)))
    avg_loss = total_loss / len(loader)
    return eval_model, avg_loss, acc


_, mean_loss, mean_acc = evaluate_checkpoint(Path("weights_mean.pth"), "mean", val_loader)
attn_model, attn_loss, attn_acc = evaluate_checkpoint(Path("weights_attention.pth"), "attention", val_loader)

print(f"Mean Pooling      -> Val Loss: {mean_loss:.5f} | Val Acc: {mean_acc * 100:.2f}%")
print(f"Attention Pooling -> Val Loss: {attn_loss:.5f} | Val Acc: {attn_acc * 100:.2f}%")

In [ ]:
for name, tensor in torch.load("weights_attention.pth", map_location="cpu", weights_only=True).items():
    print(f"{name:35s} | shape: {str(tuple(tensor.shape)):15s} | mean: {tensor.float().mean():+.4f}")

In [ ]:
%run audio-text_transformer.py --learning-rate 1e-4 --seed 42

In [ ]:
weights = torch.load(
    "transformer_weights.pth",
    map_location="cpu",
    weights_only=True
)

for name, tensor in weights.items():
    print(f"Layer: {name}")
    print(f"Shape: {tensor.shape}")
    print(f"Weights:\n{tensor}")
    print("-" * 50)

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch import nn

model = attn_model  # or: model = mean_model
model.to(DEVICE)

dataset_root = Path("/data/COMS30017")

# Load validation features
val_audio, val_text, val_labels = load_features(
    dataset_root, "val"
)

# Create validation dataset
val_dataset = AudioTextDataset(
    val_audio, val_text, val_labels, training=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False
)

# Evaluation setup
criterion = nn.CrossEntropyLoss()

model.eval()

total_loss = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for audio, text, labels in val_loader:
        audio = audio.to(DEVICE)
        text = text.to(DEVICE)
        labels = labels.to(DEVICE)

        # Get model predictions
        logits = model(audio, text)

        # Calculate loss
        loss = criterion(logits, labels)
        total_loss += loss.item()

        # Get predicted classes
        predictions = logits.argmax(dim=-1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Calculate metrics
accuracy = np.mean(
    np.array(all_preds) == np.array(all_labels)
)

average_loss = total_loss / len(val_loader)

print("========== MODEL EVALUATION ==========")
print(f"Validation Loss: {average_loss:.5f}")
print(f"Validation Accuracy: {accuracy * 100:.2f}%")
print(f"Total Samples: {len(all_labels)}")
print("======================================")
